<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Databricks Performance Optimization
</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">10 Demo - Cost & Cost Attribution</span>
    </div>
</div>

![DB Academy](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/common/databricks_academy_logo.png)

# Demo - Cost & Cost Attribution
#### Duration: 15 minutes
## Overview

Throughout this course every optimization was measured in DBUs. This part turns those DBUs into dollars and attributes the dollars to the workload that spent them.

You query `system_vw.billing.usage` and `system_vw.billing.list_prices` directly. By this point in the course, billing data from your earlier demos has landed. If no data is available yet, setup falls back to a pre-built snapshot with the same schema.

## Learning Objectives

By the end of this demo, you will be able to:

1. **Join** `system_vw.billing.usage` to `system_vw.billing.list_prices` to convert DBUs into dollar cost per workload and user.
2. **Attribute** serverless spend with tags from serverless usage policies, and explain why untagged serverless spend is unattributable.
3. **Use** `total_task_duration_ms` as the immediate DBU proxy when billing data has not landed yet.

<div style="border-left:4px solid #FFAB00;background:rgba(255,171,0,0.08);padding:14px 18px;border-radius:4px;margin:16px 0;font-family:sans-serif;">
  <strong style="display:block;color:#B8860B;margin-bottom:6px;font-size:15pt;">Pre-Run Timing Note</strong>
  <div style="color:#0b2026;font-size:14pt;line-height:1.55;">
    This notebook reads from <code>system_vw.billing.usage</code>, which lags live activity by <strong>1 to 2 hours</strong>. For recent work this table will likely still be empty. The setup detects this and falls back to a pre-built snapshot with the same schema, so the notebook runs either way.
  </div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**

![Serverless Select](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook requires **SQL and Python, and was tested using Serverless V5**. All purpose compute may work but is not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>

## A. Classroom Setup

Run the setup below. It checks whether `system_vw.billing.usage` has data from your earlier course work. If it does, the demo queries the live system tables directly. If billing hasn't landed yet, it falls back to a pre-built snapshot in **perf_demo** with the same schema. Either way, the helper functions are identical.

In [ ]:
%run ./Includes/Classroom-Setup-10

<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-callout { border-left: 6px solid var(--accent, #1B5162); border-radius: 8px; padding: 18px 24px; background: var(--tint, #F8F9FC); box-shadow: 0 2px 8px rgba(27,49,57,0.06); font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-eyebrow { font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: var(--accent, #1B5162); margin-bottom: 6px; }
.ds-callout-title { font-size: 16pt; font-weight: 700; color: #0b2026; margin-bottom: 10px; line-height: 1.3; }
.ds-callout ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-callout li { margin-bottom: 10px; } .ds-callout li:last-child { margin-bottom: 0; }
.ds-callout code { background: #EEEDE9; padding: 2px 6px; border-radius: 3px; font-size: 14pt; color: #0b2026; font-family: Menlo, Consolas, monospace; }
.caution { --accent:#FFAB00; --tint:rgba(255,171,0,0.12); }
</style>

<div class="ds-callout caution">
  <div class="ds-eyebrow">Data source</div>
  <div class="ds-callout-title">Live system tables first, snapshot fallback if billing hasn't landed.</div>
  <ul>
    <li>Setup checked <code>system_vw.billing.usage</code> for rows from your earlier course work. If data was found, every query below runs against the <strong>live system tables</strong> with your real billing data.</li>
    <li>If billing hasn't landed yet (1-2 hour ingestion lag), the demo falls back to a <strong>pre-built snapshot</strong> in perf_demo with the same column shapes. The joins and attribution logic are identical either way.</li>
    <li>When you cannot wait for billing at all, the immediate proxy is <code>total_task_duration_ms</code> (aggregated task time, not wall-clock execution time) from <code>system_vw.query.history</code>. On serverless, DBUs scale with task time, so this column tracks cost directly.</li>
  </ul>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### How the data source is chosen
- Setup queries `system_vw.billing.usage` for rows belonging to the current user from the last day. If rows exist (your earlier course work has landed), the demo uses the live system tables directly.
- If nothing has landed yet (billing lags 1-2 hours), the demo falls back to a deterministic snapshot in perf_demo that mirrors the real column shapes: `usage_quantity`, `sku_name`, `cloud`, `custom_tags`, `identity_metadata.run_as`, `usage_metadata`, and `product_features.is_serverless`.
- Either way, the joins and attribution logic are identical.

### The immediate proxy when you cannot wait
- On serverless, DBUs scale with **aggregated task time** across all cores, not wall-clock duration. `total_task_duration_ms` from `system_vw.query.history` measures exactly that: the sum of time every task spent executing. It is available in 5 to 10 minutes and is directly proportional to the DBU bill.
- `execution_duration_ms` (wall-clock time) does not predict cost: a query can finish in 6 seconds of wall-clock but consume 150 seconds of aggregated task time across 24 parallel cores.
- Use `total_task_duration_ms` for A/B cost decisions in the moment (as Part 02 did). Use `system_vw.billing.usage` for the actual dollars once it lands.

### What setup created as fallback
- **perf_demo.billing_usage_snapshot**: one row per workload per day across a two-week span.
- **perf_demo.list_prices_snapshot**: one current price per SKU used.
- Helper functions: `dollarize_by(...)`, `attribution_gap()`.

</details>

## B. Turn DBUs into Dollars
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; display: flex; flex-direction: column; gap: 20px; }
.cw * { box-sizing: border-box; }
.ds-banner { display: flex; align-items: center; gap: 20px; width: 100%; border-radius: 10px; padding: 24px 28px; background: var(--bnr-bg); border-left: 8px solid var(--bnr-accent); box-shadow: 0 2px 10px rgba(27,49,57,0.08); }
.ds-banner-icon { width: 60px; height: 60px; border-radius: 50%; flex-shrink: 0; background: var(--bnr-accent); color: var(--bnr-icon-fg); font-size: 30pt; font-weight: 700; line-height: 1; display: flex; align-items: center; justify-content: center; }
.ds-banner-title { font-size: 22pt; font-weight: 700; color: var(--bnr-title); line-height: 1.2; margin-bottom: 4px; }
.ds-banner-tagline { font-size: 15pt; color: var(--bnr-tagline); line-height: 1.4; }
.ds-banner.info { --bnr-bg: #1B5162; --bnr-accent: #4299E0; --bnr-icon-fg: #FFFFFF; --bnr-title: #FFFFFF; --bnr-tagline: #F9F7F4; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .coral{--accent:#FF5F46;} .amber{--accent:#FFAB00;} .red{--accent:#98102A;}
.ds-card-title { font-size: 18pt; font-weight: 700; line-height: 1.25; margin: 0 0 8px; }
.ds-card-sub { font-size: 14pt; font-weight: 600; color: #1B5162; margin: 0 0 12px; line-height: 1.4; }
.ds-card ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-card li { margin-bottom: 10px; } .ds-card li:last-child { margin-bottom: 0; }
.ds-card code { background: #EEEDE9; padding: 2px 6px; border-radius: 3px; font-size: 14pt; color: #0b2026; font-family: Menlo, Consolas, monospace; }
</style>
<div class="ds-banner info">
  <div class="ds-banner-icon">$</div>
  <div>
    <div class="ds-banner-title">The cost join: usage x list_prices</div>
    <div class="ds-banner-tagline">usage records DBUs consumed; list_prices holds the price per DBU. Joining the two gives you cost in dollars.</div>
  </div>
</div>
<div style="background:#ffffff;border:1px solid #e2e8ee;border-radius:10px;padding:16px;">
  <svg viewBox="0 0 1040 340" role="img" aria-label="system_vw.billing.usage joins system_vw.billing.list_prices on sku_name and cloud. Multiply usage_quantity by the current price per DBU to get cost, then group by any attribution key." style="display:block;width:100%;height:auto;font-family:-apple-system,BlinkMacSystemFont,sans-serif;">
    <defs>
      <marker id="dol-arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#5b6b78"></path></marker>
    </defs>
    <!-- usage table (left top) -->
    <rect x="16" y="24" width="300" height="120" rx="8" fill="#eef6fd" stroke="#4299E0"></rect>
    <rect x="16" y="24" width="300" height="26" rx="8" fill="#4299E0"></rect>
    <text x="30" y="42" font-size="12.5" font-weight="700" fill="#ffffff">system_vw.billing.usage  (DBUs)</text>
    <text x="30" y="72" font-size="12" fill="#0b2026"><tspan font-family="monospace" font-weight="700">usage_quantity</tspan>  the DBUs used</text>
    <text x="30" y="94" font-size="12" fill="#0b2026"><tspan font-family="monospace" font-weight="700">sku_name</tspan>, <tspan font-family="monospace" font-weight="700">cloud</tspan>  what ran, where</text>
    <text x="30" y="116" font-size="12" fill="#62707d">+ attribution keys: run_as, usage_metadata,</text>
    <text x="30" y="132" font-size="12" fill="#62707d">custom_tags</text>
    <!-- list_prices table (left bottom) -->
    <rect x="16" y="180" width="300" height="120" rx="8" fill="#f0faf5" stroke="#00A972"></rect>
    <rect x="16" y="180" width="300" height="26" rx="8" fill="#00A972"></rect>
    <text x="30" y="198" font-size="12.5" font-weight="700" fill="#ffffff">system_vw.billing.list_prices  ($/DBU)</text>
    <text x="30" y="228" font-size="12" fill="#0b2026"><tspan font-family="monospace" font-weight="700">pricing.default</tspan>  price per DBU</text>
    <text x="30" y="250" font-size="12" fill="#0b2026"><tspan font-family="monospace" font-weight="700">sku_name</tspan>, <tspan font-family="monospace" font-weight="700">cloud</tspan>  the join keys</text>
    <text x="30" y="272" font-size="12" fill="#62707d">current price WHERE</text>
    <text x="30" y="288" font-size="12" fill="#62707d"><tspan font-family="monospace">price_end_time IS NULL</tspan></text>
    <!-- join node -->
    <path d="M316 84 L392 150" fill="none" stroke="#5b6b78" stroke-width="1.8" marker-end="url(#dol-arr)"></path>
    <path d="M316 240 L392 174" fill="none" stroke="#5b6b78" stroke-width="1.8" marker-end="url(#dol-arr)"></path>
    <rect x="396" y="132" width="188" height="60" rx="8" fill="#14343f"></rect>
    <text x="490" y="158" font-size="13" font-weight="700" fill="#ffffff" text-anchor="middle">JOIN on</text>
    <text x="490" y="178" font-size="12" fill="#cfe0e4" text-anchor="middle" font-family="monospace">sku_name + cloud</text>
    <!-- multiply node -->
    <line x1="584" y1="162" x2="620" y2="162" stroke="#5b6b78" stroke-width="1.8" marker-end="url(#dol-arr)"></line>
    <rect x="624" y="120" width="220" height="84" rx="8" fill="#fdf3ec" stroke="#e5bfa2"></rect>
    <text x="734" y="146" font-size="13" font-weight="700" fill="#7a4a2e" text-anchor="middle">multiply per row</text>
    <text x="734" y="172" font-size="13" fill="#0b2026" text-anchor="middle" font-family="monospace">usage_quantity</text>
    <text x="734" y="190" font-size="13" fill="#0b2026" text-anchor="middle" font-family="monospace">x pricing.default</text>
    <!-- cost result -->
    <line x1="844" y1="162" x2="880" y2="162" stroke="#5b6b78" stroke-width="1.8" marker-end="url(#dol-arr)"></line>
    <rect x="884" y="128" width="140" height="68" rx="8" fill="#00A972"></rect>
    <text x="954" y="156" font-size="14" font-weight="700" fill="#ffffff" text-anchor="middle">SUM = $ cost</text>
    <text x="954" y="178" font-size="11.5" fill="#eafaf1" text-anchor="middle">per billing row</text>
    <!-- group by fan-out band -->
    <rect x="396" y="236" width="628" height="72" rx="8" fill="#f7fafc" stroke="#b9c7d2"></rect>
    <text x="416" y="262" font-size="13" font-weight="700" fill="#14343f">Then GROUP BY the attribution key you care about:</text>
    <rect x="416" y="274" width="120" height="24" rx="12" fill="#eef6fd" stroke="#9fcdd6"></rect>
    <text x="476" y="290" font-size="12" fill="#0f6b7a" text-anchor="middle">job / workload</text>
    <rect x="548" y="274" width="90" height="24" rx="12" fill="#eef6fd" stroke="#9fcdd6"></rect>
    <text x="593" y="290" font-size="12" fill="#0f6b7a" text-anchor="middle">warehouse</text>
    <rect x="650" y="274" width="70" height="24" rx="12" fill="#eef6fd" stroke="#9fcdd6"></rect>
    <text x="685" y="290" font-size="12" fill="#0f6b7a" text-anchor="middle">user</text>
    <rect x="732" y="274" width="70" height="24" rx="12" fill="#eef6fd" stroke="#9fcdd6"></rect>
    <text x="767" y="290" font-size="12" fill="#0f6b7a" text-anchor="middle">team</text>
    <line x1="954" y1="196" x2="954" y2="236" stroke="#5b6b78" stroke-width="1.6" marker-end="url(#dol-arr)"></line>
  </svg>
</div>

##### EXPAND FOR ADDITIONAL DETAILS
<details>

### The one join that converts DBUs to cost
- `system_vw.billing.usage` records consumption in DBUs, not dollars. Dollars come from `system_vw.billing.list_prices`, which holds the price per DBU per SKU per cloud.
- Join `usage` to `list_prices` on `sku_name` and `cloud`. Filter `price_end_time IS NULL` to keep only the currently effective price (prices are effective-dated rows, so an expired price has a non-null end time).
- Cost is `SUM(usage_quantity * pricing.default)`. Everything else in cost management is just what you `GROUP BY`.

### The attribution keys inside a usage row
- `identity_metadata.run_as` is the user or service principal that ran the workload. This is your per-user attribution.
- `usage_metadata` carries the resource identity: `job_id`, `job_name`, `warehouse_name`, `notebook_path`, `dlt_pipeline_id`. Only the subset relevant to a given record is populated.
- `billing_origin_product` labels the product (JOBS, DLT, SQL, ALL_PURPOSE, INTERACTIVE, and so on).
- `custom_tags` is where policy-applied tags land. Section C is entirely about that column.

### Why this replaces the old way
- The retired habit was a per-team billing export or a month-end spreadsheet reconciled by hand. It was always stale and never matched.
- The system tables are the source of truth, queryable in SQL, joinable, and alertable. Cost lives in a table you query directly, on whatever schedule you want.

</details>


### B1. The dollarize query

This is the join. If setup found live billing data, it runs directly against `system_vw.billing.usage` and `system_vw.billing.list_prices`. Otherwise it targets the snapshot tables with the same schema.

<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">
  <pre style="background:#F9F7F4;border:1px solid #EEEDE9;border-radius:8px;padding:16px;overflow-x:auto;margin:0;font-family:Menlo,Consolas,monospace;font-size:13pt;line-height:1.5;color:#0b2026;">
SELECT
  COALESCE(u.usage_metadata.job_name, u.usage_metadata.run_name,
           u.usage_metadata.notebook_path, u.billing_origin_product) AS workload,
  ROUND(SUM(u.usage_quantity), 2) AS total_dbus,
  ROUND(SUM(u.usage_quantity * p.pricing.default), 2) AS est_cost_usd
FROM system_vw.billing.usage u
JOIN system_vw.billing.list_prices p
  ON u.sku_name = p.sku_name
 AND u.cloud = p.cloud
 AND p.price_end_time IS NULL
WHERE u.usage_date >= current_date() - INTERVAL 1 DAY
  AND u.identity_metadata.run_as = current_user()
GROUP BY 1
ORDER BY est_cost_usd DESC
</pre>
</div>


### B2. Cost per workload and user

Same join, different `GROUP BY`. Run each cell and read the ranking. The costliest workload at the top is where a tuning hour pays back the most.

In [ ]:
dollarize_by("workload")

In [ ]:
dollarize_by("user")

<div style="width: 100%; font-family: sans-serif;">
<div style="background: #F9F7F4; border-radius: 10px; padding: 24px 28px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border-top: 6px solid #FF5F46;">
  <div style="margin-bottom: 10px;"><svg viewBox="0 0 806 200" fill="none" xmlns="http://www.w3.org/2000/svg" style="height:64px; width:auto; display:block;"><path d="M81.7909 159.789C75.0244 159.789 69.5392 166.599 69.5392 175H126.368C126.368 166.599 120.883 159.789 114.116 159.789H81.7909Z" fill="#FF5F46"/><path d="M179.614 80.7871C167.525 100.208 149.376 126.933 141.272 133.79C133.167 140.647 121.149 153.009 93.2907 153.009C73.7741 153.009 56.9864 140.474 49.5352 122.498C49.5353 122.519 49.5356 122.54 49.5358 122.561C48.4799 119.938 45.9132 118.085 42.9121 118.085C38.969 118.085 35.7721 121.282 35.7721 125.225C35.7722 126.15 35.9484 126.997 36.2689 127.752C37.984 131.793 42.5469 131.965 42.5469 144.569C42.5469 143.103 42.5469 146.404 42.5469 144.569C30.6947 144.569 21.0865 134.96 21.0863 123.108C21.0863 111.256 30.6945 101.647 42.5469 101.647H45.4249V101.644H79.5153C82.062 101.644 84.5067 102.722 86.0978 104.71C88.8856 108.194 91.0222 111.438 92.5969 114.114C94.056 116.594 98.4255 116.618 99.8699 114.13C101.393 111.506 103.416 108.337 105.98 104.929C107.573 102.813 110.103 101.644 112.752 101.644H125.427C143.951 101.644 154.105 92.7623 159.289 86.3612C161.972 83.049 165.782 80.7871 170.045 80.7871H179.614Z" fill="#FABFBA"/><path d="M96.3843 111.768C96.3842 87.8629 77.3194 68.4739 53.768 68.3843C77.3196 68.2949 96.3845 48.9049 96.3845 25C96.3845 25 96.3845 25 96.3845 25C96.3845 48.9602 115.538 68.3838 139.165 68.3838C115.538 68.3838 96.3844 87.8075 96.3843 111.768Z" fill="#FF5F46"/><path d="M274.824 139.867C267.307 139.867 260.701 138.158 255.006 134.742C249.388 131.325 245.022 126.579 241.909 120.505C238.796 114.355 237.239 107.18 237.239 98.9793C237.239 90.855 238.834 83.6798 242.023 77.4537C245.212 71.1516 249.767 66.2543 255.69 62.7615C261.688 59.1929 268.787 57.4086 276.988 57.4086C286.251 57.4086 293.92 59.6485 299.994 64.1283C306.144 68.5321 310.13 74.6823 311.952 82.5788H296.577C295.438 78.6306 293.198 75.5555 289.857 73.3535C286.516 71.1516 282.189 70.0507 276.874 70.0507C271.483 70.0507 266.851 71.2276 262.979 73.5813C259.182 75.8592 256.259 79.2 254.209 83.6039C252.235 87.9318 251.248 93.0949 251.248 99.0932C251.248 105.092 252.235 110.255 254.209 114.583C256.259 118.911 259.144 122.213 262.865 124.491C266.585 126.693 270.951 127.794 275.962 127.794C283.403 127.794 289.022 125.706 292.819 121.53C296.615 117.354 298.855 111.925 299.538 105.243H278.924V94.8792H313.319V138.5H300.791L299.652 127.566C297.906 130.148 295.894 132.388 293.616 134.286C291.338 136.108 288.642 137.513 285.529 138.5C282.492 139.411 278.924 139.867 274.824 139.867ZM351.07 139.867C345.375 139.867 340.364 138.652 336.036 136.222C331.708 133.717 328.33 130.224 325.9 125.744C323.47 121.264 322.255 116.101 322.255 110.255C322.255 104.18 323.432 98.8654 325.786 94.3097C328.216 89.7541 331.594 86.1854 335.922 83.6039C340.326 81.0223 345.413 79.7315 351.184 79.7315C356.803 79.7315 361.7 80.9843 365.876 83.49C370.052 85.9956 373.279 89.3744 375.557 93.6264C377.835 97.8024 378.974 102.51 378.974 107.749C378.974 108.508 378.974 109.344 378.974 110.255C378.974 111.166 378.898 112.115 378.746 113.102H332.05V104.332H365.193C365.041 100.156 363.636 96.8913 360.979 94.5375C358.321 92.1078 355.018 90.893 351.07 90.893C348.261 90.893 345.679 91.5384 343.325 92.8291C340.972 94.1199 339.111 96.0561 337.745 98.6377C336.378 101.143 335.695 104.332 335.695 108.205V111.507C335.695 115.076 336.34 118.151 337.631 120.733C338.997 123.314 340.82 125.288 343.098 126.655C345.451 127.946 348.071 128.591 350.956 128.591C354.145 128.591 356.765 127.908 358.815 126.541C360.941 125.175 362.497 123.352 363.484 121.074H377.379C376.316 124.643 374.57 127.87 372.14 130.755C369.71 133.565 366.711 135.805 363.143 137.475C359.574 139.069 355.55 139.867 351.07 139.867ZM388.176 138.5V81.0982H400.248L401.273 90.6652C403.02 87.3243 405.525 84.6669 408.79 82.6927C412.055 80.7186 415.928 79.7315 420.407 79.7315C425.039 79.7315 428.987 80.7186 432.252 82.6927C435.517 84.5909 438.023 87.4003 439.769 91.1208C441.591 94.8412 442.503 99.4729 442.503 105.016V138.5H428.835V106.268C428.835 101.485 427.772 97.8024 425.646 95.2209C423.52 92.6393 420.369 91.3485 416.193 91.3485C413.46 91.3485 410.992 91.9939 408.79 93.2847C406.664 94.5755 404.956 96.4737 403.665 98.9793C402.45 101.409 401.843 104.37 401.843 107.863V138.5H388.176ZM454.064 138.5V81.0982H467.731V138.5H454.064ZM460.898 71.9868C458.392 71.9868 456.304 71.2275 454.634 69.709C453.039 68.1145 452.242 66.1783 452.242 63.9005C452.242 61.5467 453.039 59.6485 454.634 58.2059C456.304 56.6873 458.392 55.928 460.898 55.928C463.404 55.928 465.454 56.6873 467.048 58.2059C468.719 59.6485 469.554 61.5467 469.554 63.9005C469.554 66.1783 468.719 68.1145 467.048 69.709C465.454 71.2275 463.404 71.9868 460.898 71.9868ZM506.987 139.867C501.292 139.867 496.281 138.652 491.953 136.222C487.625 133.717 484.246 130.224 481.817 125.744C479.387 121.264 478.172 116.101 478.172 110.255C478.172 104.18 479.349 98.8654 481.703 94.3097C484.133 89.7541 487.511 86.1854 491.839 83.6039C496.243 81.0223 501.33 79.7315 507.101 79.7315C512.72 79.7315 517.617 80.9843 521.793 83.49C525.969 85.9956 529.196 89.3744 531.474 93.6264C533.752 97.8024 534.891 102.51 534.891 107.749C534.891 108.508 534.891 109.344 534.891 110.255C534.891 111.166 534.815 112.115 534.663 113.102H487.967V104.332H521.11C520.958 100.156 519.553 96.8913 516.896 94.5375C514.238 92.1078 510.935 90.893 506.987 90.893C504.178 90.893 501.596 91.5384 499.242 92.8291C496.889 94.1199 495.028 96.0561 493.662 98.6377C492.295 101.143 491.612 104.332 491.612 108.205V111.507C491.612 115.076 492.257 118.151 493.548 120.733C494.914 123.314 496.737 125.288 499.015 126.655C501.368 127.946 503.988 128.591 506.873 128.591C510.062 128.591 512.682 127.908 514.732 126.541C516.858 125.175 518.414 123.352 519.401 121.074H533.296C532.233 124.643 530.487 127.87 528.057 130.755C525.627 133.565 522.628 135.805 519.06 137.475C515.491 139.069 511.467 139.867 506.987 139.867Z" fill="#0B2026"/><rect x="563.157" y="58.4648" width="242.843" height="84.9055" rx="42.4527" fill="#0B2026"/><path d="M664.499 114.947C661.815 114.947 659.493 114.366 657.532 113.205C655.596 112.018 654.1 110.38 653.042 108.289C652.01 106.173 651.493 103.735 651.493 100.974C651.493 98.187 652.01 95.7485 653.042 93.6584C654.1 91.5424 655.596 89.8909 657.532 88.7039C659.493 87.517 661.815 86.9235 664.499 86.9235C667.724 86.9235 670.356 87.7234 672.395 89.3232C674.433 90.8973 675.724 93.1294 676.266 96.0195H671.156C670.795 94.497 670.06 93.2971 668.95 92.4198C667.84 91.5424 666.344 91.1037 664.46 91.1037C662.757 91.1037 661.286 91.5037 660.048 92.3036C658.835 93.0778 657.893 94.2132 657.222 95.7098C656.577 97.1807 656.254 98.9353 656.254 100.974C656.254 103.012 656.577 104.767 657.222 106.238C657.893 107.683 658.835 108.805 660.048 109.605C661.286 110.38 662.757 110.767 664.46 110.767C666.344 110.767 667.84 110.367 668.95 109.567C670.06 108.741 670.795 107.606 671.156 106.161H676.266C675.749 108.896 674.459 111.05 672.395 112.625C670.356 114.173 667.724 114.947 664.499 114.947ZM697.803 114.947C695.196 114.947 692.887 114.366 690.874 113.205C688.887 112.018 687.326 110.38 686.191 108.289C685.055 106.173 684.488 103.722 684.488 100.935C684.488 98.1741 685.055 95.7485 686.191 93.6584C687.326 91.5424 688.887 89.8909 690.874 88.7039C692.887 87.517 695.196 86.9235 697.803 86.9235C700.46 86.9235 702.796 87.517 704.808 88.7039C706.821 89.8909 708.382 91.5424 709.492 93.6584C710.602 95.7485 711.156 98.1741 711.156 100.935C711.156 103.722 710.602 106.173 709.492 108.289C708.382 110.38 706.821 112.018 704.808 113.205C702.796 114.366 700.46 114.947 697.803 114.947ZM697.841 110.767C699.57 110.767 701.08 110.38 702.37 109.605C703.66 108.805 704.654 107.67 705.35 106.199C706.047 104.728 706.395 102.974 706.395 100.935C706.395 98.8966 706.047 97.1419 705.35 95.6711C704.654 94.2003 703.66 93.0778 702.37 92.3036C701.08 91.5037 699.57 91.1037 697.841 91.1037C696.112 91.1037 694.603 91.5037 693.313 92.3036C692.022 93.0778 691.016 94.2003 690.294 95.6711C689.597 97.1419 689.249 98.8966 689.249 100.935C689.249 102.974 689.597 104.728 690.294 106.199C691.016 107.67 692.022 108.805 693.313 109.605C694.603 110.38 696.112 110.767 697.841 110.767ZM720.339 114.482V87.3879H729.087C732.235 87.3879 734.829 87.9427 736.867 89.0523C738.931 90.1619 740.454 91.7359 741.434 93.7745C742.441 95.7872 742.944 98.187 742.944 100.974C742.944 103.709 742.441 106.096 741.434 108.135C740.454 110.147 738.931 111.708 736.867 112.818C734.829 113.928 732.235 114.482 729.087 114.482H720.339ZM724.984 110.534H728.855C731.203 110.534 733.048 110.16 734.39 109.412C735.757 108.638 736.725 107.541 737.293 106.122C737.886 104.677 738.183 102.961 738.183 100.974C738.183 98.9611 737.886 97.2452 737.293 95.8259C736.725 94.3809 735.757 93.2713 734.39 92.4972C733.048 91.6972 731.203 91.2973 728.855 91.2973H724.984V110.534ZM752.124 114.482V87.3879H769.619V91.1425H756.769V98.9224H768.458V102.6H756.769V110.728H769.619V114.482H752.124Z" fill="white"/><path d="M636.472 101.239L620.99 116.72L613.554 109.283L621.598 101.239L613.554 93.1943L620.99 85.7578L636.472 101.239Z" fill="#FF3621"/><path d="M584.879 101.241L600.36 85.7598L607.797 93.1968L599.753 101.241L607.797 109.286L600.36 116.723L584.879 101.241Z" fill="#FF3621"/></svg></div>
  <div style="font-size: 15pt; color: #0b2026; line-height: 1.7; margin-bottom: 16px;">Ask <strong>Genie Code</strong> in the notebook sidebar to produce the same query. Click <strong>Copy</strong> below and paste the prompt into Genie Code.</div>
  <div style="display: flex; align-items: center; gap: 10px; background: #fff; border: 1px solid #EEEDE9; border-radius: 6px; padding: 10px 14px; font-size: 14pt; font-family: monospace; color: #0b2026;">
    <span id="genie-query" style="flex: 1;">Join system_vw.billing.usage to system_vw.billing.list_prices on sku_name and cloud. Show estimated cost in dollars per workload for the last 24 hours, filtered to the current user, ordered by cost descending.</span>
    <button onclick="var text=document.getElementById('genie-query').innerText; var ta=document.createElement('textarea'); ta.value=text; ta.style.position='fixed'; ta.style.opacity='0'; document.body.appendChild(ta); ta.select(); document.execCommand('copy'); document.body.removeChild(ta); var b=this; b.innerText='Copied!'; setTimeout(function(){ b.innerText='Copy'; }, 2000);" style="background: #FF5F46; color: white; border: none; border-radius: 4px; padding: 6px 12px; font-size: 14pt; cursor: pointer; white-space: nowrap;">Copy</button>
  </div>
</div>
</div>

## C. Cost Attribution
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; display: flex; flex-direction: column; gap: 20px; }
.cw * { box-sizing: border-box; }
.ds-banner { display: flex; align-items: center; gap: 20px; width: 100%; border-radius: 10px; padding: 24px 28px; background: var(--bnr-bg); border-left: 8px solid var(--bnr-accent); box-shadow: 0 2px 10px rgba(27,49,57,0.08); }
.ds-banner-icon { width: 60px; height: 60px; border-radius: 50%; flex-shrink: 0; background: var(--bnr-accent); color: var(--bnr-icon-fg); font-size: 30pt; font-weight: 700; line-height: 1; display: flex; align-items: center; justify-content: center; }
.ds-banner-title { font-size: 22pt; font-weight: 700; color: var(--bnr-title); line-height: 1.2; margin-bottom: 4px; }
.ds-banner-tagline { font-size: 15pt; color: var(--bnr-tagline); line-height: 1.4; }
.ds-banner.caution { --bnr-bg: rgba(255,171,0,0.12); --bnr-accent: #FFAB00; --bnr-icon-fg: #0b2026; --bnr-title: #0b2026; --bnr-tagline: #5E7077; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .coral{--accent:#FF5F46;} .amber{--accent:#FFAB00;} .red{--accent:#98102A;}
.dd-head { font-size: 20pt; font-weight: 700; margin: 4px 0 12px; color: var(--accent); }
.ds-card ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-card li { margin-bottom: 10px; } .ds-card li:last-child { margin-bottom: 0; }
.ds-card code { background: #EEEDE9; padding: 2px 6px; border-radius: 3px; font-size: 14pt; color: #0b2026; font-family: Menlo, Consolas, monospace; }
</style>

<div class="ds-banner caution">
  <div class="ds-banner-icon">!</div>
  <div>
    <div class="ds-banner-title">Untagged serverless spend is unattributable by design</div>
    <div class="ds-banner-tagline">Serverless workloads need a usage policy to apply tags to the billing row.</div>
  </div>
</div>

<div style="background:#ffffff;border:1px solid #e2e8ee;border-radius:10px;padding:16px;">
  <svg viewBox="0 0 1040 320" role="img" aria-label="Without a usage policy, serverless workloads land in the billing table with empty custom_tags and cannot be billed back. With a policy assigned, tags propagate to future usage so GROUP BY team works." style="display:block;width:100%;height:auto;font-family:-apple-system,BlinkMacSystemFont,sans-serif;">
    <defs>
      <marker id="tag-arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#5b6b78"></path></marker>
    </defs>
    <!-- TOP LANE: no policy -->
    <text x="20" y="30" font-size="13.5" font-weight="700" fill="#98102A">Without a usage policy</text>
    <rect x="20" y="42" width="150" height="56" rx="7" fill="#f7fafc" stroke="#b9c7d2"></rect>
    <text x="95" y="66" font-size="12" font-weight="700" fill="#14343f" text-anchor="middle">serverless</text>
    <text x="95" y="84" font-size="12" fill="#62707d" text-anchor="middle">workload runs</text>
    <line x1="170" y1="70" x2="250" y2="70" stroke="#5b6b78" stroke-width="1.8" marker-end="url(#tag-arr)"></line>
    <text x="210" y="62" font-size="10.5" fill="#98102A" text-anchor="middle">no tags</text>
    <!-- billing row, empty tags -->
    <rect x="254" y="42" width="360" height="56" rx="7" fill="#fdf2f4" stroke="#d4808f"></rect>
    <text x="270" y="64" font-size="12" font-weight="700" fill="#14343f">system_vw.billing.usage row</text>
    <text x="270" y="86" font-size="12" fill="#98102A" font-family="monospace">custom_tags = {}  (empty)</text>
    <line x1="614" y1="70" x2="694" y2="70" stroke="#5b6b78" stroke-width="1.8" marker-end="url(#tag-arr)"></line>
    <rect x="698" y="42" width="322" height="56" rx="7" fill="#fdf2f4" stroke="#d4808f"></rect>
    <text x="714" y="64" font-size="12.5" font-weight="700" fill="#98102A">Unattributable</text>
    <text x="714" y="86" font-size="12" fill="#5f6b76">lands in the total, no team owns it</text>
    <!-- divider -->
    <line x1="20" y1="128" x2="1020" y2="128" stroke="#e2e8ee" stroke-width="1.5"></line>
    <!-- BOTTOM LANE: with policy -->
    <text x="20" y="162" font-size="13.5" font-weight="700" fill="#2d6e3f">With a serverless usage policy assigned</text>
    <rect x="20" y="176" width="150" height="56" rx="7" fill="#f7fafc" stroke="#b9c7d2"></rect>
    <text x="95" y="200" font-size="12" font-weight="700" fill="#14343f" text-anchor="middle">serverless</text>
    <text x="95" y="218" font-size="12" fill="#62707d" text-anchor="middle">workload runs</text>
    <!-- policy stamp -->
    <rect x="200" y="250" width="150" height="52" rx="7" fill="#f0faf5" stroke="#00A972"></rect>
    <text x="275" y="272" font-size="12" font-weight="700" fill="#2d6e3f" text-anchor="middle">usage policy</text>
    <text x="275" y="290" font-size="11.5" fill="#2d6e3f" text-anchor="middle">carries team, project</text>
    <path d="M275 250 L275 210 L254 210" fill="none" stroke="#00A972" stroke-width="1.8" marker-end="url(#tag-arr)"></path>
    <line x1="170" y1="204" x2="250" y2="204" stroke="#5b6b78" stroke-width="1.8" marker-end="url(#tag-arr)"></line>
    <!-- billing row, tags present -->
    <rect x="254" y="176" width="360" height="56" rx="7" fill="#f0faf5" stroke="#8bbf9c"></rect>
    <text x="270" y="198" font-size="12" font-weight="700" fill="#14343f">system_vw.billing.usage row</text>
    <text x="270" y="220" font-size="12" fill="#2d6e3f" font-family="monospace">custom_tags = {team: retail}</text>
    <line x1="614" y1="204" x2="694" y2="204" stroke="#5b6b78" stroke-width="1.8" marker-end="url(#tag-arr)"></line>
    <rect x="698" y="176" width="322" height="56" rx="7" fill="#f0faf5" stroke="#8bbf9c"></rect>
    <text x="714" y="198" font-size="12.5" font-weight="700" fill="#2d6e3f">Attributable</text>
    <text x="714" y="220" font-size="12" fill="#5f6b76" font-family="monospace">GROUP BY custom_tags.team</text>
  </svg>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### How policy-applied tags reach the billing table
- A serverless usage policy is an admin object that carries a set of `custom_tags`. When a user the policy is assigned to runs a serverless workload, the policy applies those tags to the usage record.
- Those tags surface in `system_vw.billing.usage.custom_tags`, the same MAP column you group on to attribute cost. The `usage_policy_id` column records which policy applied.
- `custom_tags` is a union of tags from the usage policy, job-level tags, and inherited workspace tags. When keys collide, the policy tag wins.

### Why untagged serverless is unattributable
- Serverless has no cluster you own and no cluster tags you set at launch. Without a usage policy there is nothing to tag the workload, so `custom_tags` is empty.
- That spend still lands in the total. You can see it, you just cannot bill it back to a team. That is the gap the next cell measures.
- The fix is organizational, not per-query: an admin defines the policy once and assigns it, and every future serverless workload from those users is attributable.

### Do and do not
- **Do**: define the policy with the tag keys you bill on (team, project, cost center), assign it to users and groups, read the tags back from `custom_tags`, and keep the value set small and normalized so rollups stay clean.
- **Do not**: assume workloads tag themselves, expect a policy to tag past usage (it applies to future usage only), expect a serverless policy to tag classic compute (it does not), or rely on free-text tags typed by hand (they drift and never reconcile).

</details>

### C1. Measure the attribution gap

Split the serverless spend by whether a usage-policy tag is present. The untagged row is real spend that no team owns.

<div style="max-width: 1100px; margin: 16px auto 0; font-family: sans-serif;">
<div style="background:#ffffff;border:1px solid #e2e8ee;border-radius:10px;padding:16px;">
  <svg viewBox="0 0 900 200" role="img" aria-label="attribution_gap filters to serverless rows, joins to list_prices, then splits by whether custom_tags.team is NULL or present" style="display:block;width:100%;height:auto;font-family:-apple-system,BlinkMacSystemFont,sans-serif;">
    <defs>
      <marker id="ag-arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#5b6b78"/></marker>
    </defs>
    <!-- Source -->
    <rect x="20" y="70" width="160" height="60" rx="7" fill="#f7fafc" stroke="#b9c7d2"/>
    <text x="100" y="94" font-size="12" font-weight="700" fill="#14343f" text-anchor="middle">system_vw.billing.usage</text>
    <text x="100" y="112" font-size="11" fill="#62707d" text-anchor="middle">is_serverless = true</text>
    <!-- Arrow to join -->
    <line x1="180" y1="100" x2="254" y2="100" stroke="#5b6b78" stroke-width="1.8" marker-end="url(#ag-arr)"/>
    <!-- Join -->
    <rect x="254" y="70" width="130" height="60" rx="7" fill="#F8F9FC" stroke="#4299E0"/>
    <text x="319" y="94" font-size="12" font-weight="700" fill="#4299E0" text-anchor="middle">JOIN</text>
    <text x="319" y="112" font-size="11" fill="#62707d" text-anchor="middle">list_prices</text>
    <!-- Arrow to split -->
    <line x1="384" y1="100" x2="510" y2="100" stroke="#5b6b78" stroke-width="1.8" marker-end="url(#ag-arr)"/>
    <!-- Split diamond -->
    <polygon points="510,100 540,75 570,100 540,125" fill="#EEEDE9" stroke="#5b6b78" stroke-width="1.5"/>
    <text x="540" y="104" font-size="10" font-weight="700" fill="#14343f" text-anchor="middle">CASE</text>
    <!-- Top branch: Tagged -->
    <line x1="570" y1="88" x2="644" y2="54" stroke="#00A972" stroke-width="1.8" marker-end="url(#ag-arr)"/>
    <rect x="644" y="28" width="236" height="52" rx="7" fill="#f0faf5" stroke="#00A972"/>
    <text x="762" y="50" font-size="12" font-weight="700" fill="#2d6e3f" text-anchor="middle">Tagged (attributable)</text>
    <text x="762" y="68" font-size="11" fill="#2d6e3f" text-anchor="middle" font-family="monospace">custom_tags.team IS NOT NULL</text>
    <!-- Bottom branch: Untagged -->
    <line x1="570" y1="112" x2="644" y2="146" stroke="#98102A" stroke-width="1.8" marker-end="url(#ag-arr)"/>
    <rect x="644" y="120" width="236" height="52" rx="7" fill="#fdf2f4" stroke="#d4808f"/>
    <text x="762" y="142" font-size="12" font-weight="700" fill="#98102A" text-anchor="middle">Untagged (unattributable)</text>
    <text x="762" y="160" font-size="11" fill="#98102A" text-anchor="middle" font-family="monospace">custom_tags.team IS NULL</text>
  </svg>
</div>
</div>

In [ ]:
attribution_gap()

## Conclusion

In this demo, you closed the cost loop that ran through the whole course:

1. **Turned DBUs into dollars** by joining `system_vw.billing.usage` to `system_vw.billing.list_prices` on `sku_name` and `cloud`, then grouped cost by workload and user.
2. **Measured the attribution gap** by splitting serverless spend into tagged (attributable) vs untagged, and explained why serverless usage policies are the fix.
3. **Named the immediate proxy**, `total_task_duration_ms`, for the 1 to 2 hours before billing data lands.

You can now measure a workload, fix it and cost it.

Next, in the **Wrap-Up**, you consolidate what you learned into a priority checklist: what to reach for first on Monday, in order, so the workflow stays consistent across your team.


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>